## Walmart Recruiting - Store Sales Forecasting — Temporal Fusion Transformer (TFT)

**TFT (Temporal Fusion Transformer)**, ჩვენს N-BEATS notebook-ისგან განსხვავებით,
**multivariate** მოდელია — მას შეუძლია სამიზნე ცვლადის (`y`) გარდა გამოიყენოს
**გარე (exogenous) ცვლადებიც**. TFT აგებულია რამდენიმე კომპონენტისგან:
**Variable Selection Networks** (ავტომატურად სწავლობს რომელი input ცვლადია
თითოეულ დროის მომენტში მნიშვნელოვანი), **LSTM encoder/decoder** (ლოკალური,
თანმიმდევრობითი ნიმუშების დასაჭერად), **multi-head interpretable attention**
(გრძელვადიანი დამოკიდებულებების დასაჭერად, ერთდროულად ინტერპრეტირებადი
attention წონებით) და **static covariate encoders** (სერიის დონეზე უცვლელი
მახასიათებლების, მაგ. მაღაზიის ტიპის, ინტეგრირებისთვის).

ეს არქიტექტურული განსხვავება პირდაპირ აისახება Notebook-ის სტრუქტურაზეც:
N-BEATS-ის **Feature Selection სექცია იყო მხოლოდ საინფორმაციო** (საბაზისო
NBEATS-ს გარე ცვლადების მიღება არ შეუძლია), ხოლო აქ **იგივე selection რეალურად
განსაზღვრავს მოდელის input-ს** — შერჩეული ცვლადები ეძლევა TFT-ს, როგორც
`futr_exog_list` (მომავალშიც ცნობილი ცვლადები — `features.csv` მოცემულია
test-პერიოდის თარიღებისთვისაც) და `stat_exog_list` (მაღაზიის დონის უცვლელი
მახასიათებლები: `Type`, `Size`).

**სტრუქტურა** იმეორებს N-BEATS notebook-ის სტრუქტურას პირდაპირი შედარებისთვის:
Setup → Data Cleaning → Feature Engineering → Feature Selection (**ამჯერად
რეალურად გამოიყენება**) → Train/Validation split + Static/Future Exog ცხრილების
აგება → **HP Search Stage 1 (Random Search)** → **HP Search Stage 2 (Full
Factorial Grid Search, დინამიურად დაზუსტებული Stage 1-ის შედეგებზე დაყრდნობით)**
→ Final Training → Pipeline + Model Registry (fallback-ით უცნობი სერიებისთვის)
→ Visualizations → Full Text Summary.

**გამოთვლითი შენიშვნა:** TFT მნიშვნელოვნად უფრო მძიმეა ვიდრე NBEATS (attention +
LSTM ყოველ ბიჯზე, არა მხოლოდ fully-connected ბლოკები), ამიტომ HP search-ის
ბიუჯეტი აქ განზრახ ოდნავ პატარაა NBEATS-თან შედარებით (Stage 1: 8 trial 36-დან
ნაცვლად 10-ისა). **Stage 1-ის დასრულების შემდეგ გადახედეთ `results_df`-ს** — თუ
რომელიმე მნიშვნელობა (`input_size`, `hidden_size`, `learning_rate`,
`batch_size`) აშკარად სჯობს დანარჩენებს (როგორც NBEATS-ში `input_size=39`
დომინირებდა), საჭიროებისამებრ დაარედაქტირეთ Stage 2-ის `search_space_2`
(`input_size`-ის ნაწილი უკვე ავტომატურად/დინამიურად ითვლება Stage 1-ის
საუკეთესო შედეგებიდან).

## 0. Setup

In [1]:
!pip install -q neuralforecast wandb

In [2]:
import time
import numpy as np
import pandas as pd

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE

import wandb

WANDB_PROJECT = "walmart-sales-forecasting"
MODEL_NAME = "TFT"
wandb.login()

import glob, os
_train_matches = glob.glob("/kaggle/input/**/train.csv*", recursive=True)
DATA_DIR = os.path.dirname(_train_matches[0])
print("DATA_DIR ->", DATA_DIR, "| found:", _train_matches)

HORIZON = 39          # test.csv-ის კვირების რაოდენობა (2012-11-02 -> 2013-07-26)
VAL_SIZE = HORIZON    # early stopping-ის შიდა validation window (nf.fit-ს ესაჭიროება)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sgurj22 (TwoMusketeers) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


DATA_DIR -> /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting | found: ['/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip']


## 1. Data Cleaning
ჩატვირთვა, merge (`stores`/`features`), დუბლიკატების მოცილება, MarkDown NA->0,
CPI/Unemployment forward/back-fill თითო Store-ისთვის, უარყოფითი გაყიდვების ჭრა 0-ზე.
იდენტურია NBEATS notebook-ის (იგივე მონაცემი, იგივე გასუფთავების ლოგიკა —
სამართლიანი შედარებისთვის მოდელებს შორის).

In [3]:
train_raw = pd.read_csv(glob.glob(f"{DATA_DIR}/train.csv*")[0], parse_dates=["Date"])
test_raw = pd.read_csv(glob.glob(f"{DATA_DIR}/test.csv*")[0], parse_dates=["Date"])
stores = pd.read_csv(glob.glob(f"{DATA_DIR}/stores.csv*")[0])
features = pd.read_csv(glob.glob(f"{DATA_DIR}/features.csv*")[0], parse_dates=["Date"])

def merge_sources(df):
    feat = features.drop(columns=["IsHoliday"])  # features.csv-ს დუბლირებული IsHoliday აქვს
    return df.merge(stores, on="Store", how="left").merge(feat, on=["Store", "Date"], how="left")

def clean_data(df, is_train=True):
    df = df.copy()
    n_before = len(df)
    df = df.drop_duplicates()

    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
    markdown_na = int(df[markdown_cols].isna().sum().sum())
    df[markdown_cols] = df[markdown_cols].fillna(0).clip(lower=0)

    df = df.sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].ffill().bfill()

    neg_count = 0
    if is_train:
        neg_count = int((df["Weekly_Sales"] < 0).sum())
        df["Weekly_Sales"] = df["Weekly_Sales"].clip(lower=0)

    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    df["Store"] = df["Store"].astype(int)
    df["Dept"] = df["Dept"].astype(int)

    stats = {"rows_before": n_before, "rows_after": len(df),
              "duplicates_removed": n_before - len(df),
              "negative_sales_clipped": neg_count, "markdown_na_filled": markdown_na}
    return df, stats

train_clean, train_stats = clean_data(merge_sources(train_raw), is_train=True)
test_clean, test_stats = clean_data(merge_sources(test_raw), is_train=False)
print("train:", train_clean.shape, train_stats)
print("test:", test_clean.shape, test_stats)

train: (421570, 16) {'rows_before': 421570, 'rows_after': 421570, 'duplicates_removed': 0, 'negative_sales_clipped': 1285, 'markdown_na_filled': 1422431}
test: (115064, 15) {'rows_before': 115064, 'rows_after': 115064, 'duplicates_removed': 0, 'negative_sales_clipped': 0, 'markdown_na_filled': 51493}


In [4]:
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="cleaning", name=f"{MODEL_NAME}_Cleaning")
wandb.config.update({f"train_{k}": v for k, v in train_stats.items()})
wandb.config.update({f"test_{k}": v for k, v in test_stats.items()})
wandb.log({"train_missing_after_clean": int(train_clean.isna().sum().sum()),
           "test_missing_after_clean": int(test_clean.isna().sum().sum())})
run.finish()

test_missing_after_clean,▁
train_missing_after_clean,▁
test_missing_after_clean,0
train_missing_after_clean,0


## 2. Feature Engineering
კალენდარული ფიჩერები + Kaggle-ის ოფიციალური "special" holiday კვირები (x5 წონა
WMAE-ში) + `unique_id`/`ds`/`y`. დამატებით ვაგებთ **store-level exog ცხრილს**
(`store_exog`) `features.csv`-დან, რომელიც (მნიშვნელოვანია!) **test-პერიოდის
თარიღებსაც ფარავს** — ეს არის ის, რაც TFT-ს საშუალებას აძლევს, გამოიყენოს
"მომავალში ცნობილი" გარე ცვლადები (`futr_exog_list`) prediction window-ისთვის.

In [5]:
def add_calendar_flags(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

    superbowl = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
    labor_day = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
    thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
    christmas = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])
    df["IsSuperBowl"] = df["Date"].isin(superbowl)
    df["IsLaborDay"] = df["Date"].isin(labor_day)
    df["IsThanksgiving"] = df["Date"].isin(thanksgiving)
    df["IsChristmas"] = df["Date"].isin(christmas)
    return df

def engineer_features(df):
    df = add_calendar_flags(df)
    df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
    df["ds"] = df["Date"]
    return df

train_fe = engineer_features(train_clean)
train_fe["y"] = train_fe["Weekly_Sales"]
test_fe = engineer_features(test_clean)
print(train_fe.shape, "| unique series:", train_fe["unique_id"].nunique())

(421570, 26) | unique series: 3331


In [6]:
# Store-level exog ცხრილი features.csv-დან -- ამოწმებს, რომ ცნობილია test-პერიოდისთვისაც
def clean_store_exog(df):
    df = df.copy()
    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
    df[markdown_cols] = df[markdown_cols].fillna(0).clip(lower=0)
    df = df.sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].ffill().bfill()
    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    df["Store"] = df["Store"].astype(int)
    return df

store_exog = features.merge(stores, on="Store", how="left")
store_exog = clean_store_exog(store_exog)
store_exog = add_calendar_flags(store_exog)
store_exog = store_exog.rename(columns={"Date": "ds"})

print("store_exog date range:", store_exog["ds"].min().date(), "->", store_exog["ds"].max().date())
print("test_fe date range:   ", test_fe["ds"].min().date(), "->", test_fe["ds"].max().date())
assert store_exog["ds"].max() >= test_fe["ds"].max(), (
    "features.csv არ ფარავს მთელ test-პერიოდს -- futr_exog_list ვერ აიგება ბოლო კვირებისთვის!"
)

store_exog date range: 2010-02-05 -> 2013-07-26
test_fe date range:    2012-11-02 -> 2013-07-26


In [7]:
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="feature_engineering", name=f"{MODEL_NAME}_FeatureEngineering")
wandb.log({"n_unique_series": train_fe["unique_id"].nunique(), "n_rows_train": len(train_fe),
           "store_exog_max_date_covers_test": bool(store_exog["ds"].max() >= test_fe["ds"].max())})
run.finish()

n_rows_train,▁
n_unique_series,▁
n_rows_train,421570
n_unique_series,3331
store_exog_max_date_covers_test,True


## 3. Feature Selection (ამჯერად რეალურად გამოიყენება)

NBEATS-ისგან განსხვავებით, TFT-ს **შეუძლია** გარე ცვლადების მიღება, ამიტომ ეს
სექცია აქ განსაზღვრავს რეალურ model input-ს, არა მხოლოდ საინფორმაციო ცხრილს:

- **`futr_exog_list`**: უწყვეტი ცვლადები, რომელთა კორელაცია `y`-სთან
  აღემატება threshold-ს (იგივე კრიტერიუმი, რაც NBEATS notebook-ში იყო
  საინფორმაციო) + სტრუქტურული holiday/calendar ცვლადები (`IsHoliday`,
  `IsSuperBowl`, `IsLaborDay`, `IsThanksgiving`, `IsChristmas`, `Month`,
  `WeekOfYear`) — ეს უკანასკნელნი corr-ის threshold-ს არ ექვემდებარებიან,
  რადგან ისინი კონკურსის officially-განსაზღვრული structural სიგნალებია
  (და `IsHoliday` ასევე გამოიყენება WMAE-ის x5 წონისთვის).
- **`stat_exog_list`**: მაღაზიის დონის უცვლელი მახასიათებლები (`Type`
  one-hot-encoded, `Size` სტანდარტიზებული) — TFT-ის static covariate
  encoder-ისთვის.

ორივე ცვლადთა ჯგუფი **`features.csv`-დან მომდინარეობს, რომელიც ცნობილია
test-პერიოდისთვისაც** — ამიტომ ორივე ჯდება `futr_exog_list`-ში (არა
`hist_exog_list`-ში, რომელიც მხოლოდ წარსულისთვის ცნობილ ცვლადებს ეთმობა).

In [8]:
candidate_exog = ["Temperature", "Fuel_Price", "CPI", "Unemployment",
                   "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

corr = train_fe[candidate_exog + ["y"]].corr()["y"].drop("y").abs()
missing_rate = train_fe[candidate_exog].isna().mean()
selection_table = pd.DataFrame({"abs_corr_with_y": corr, "missing_rate": missing_rate}) \
    .sort_values("abs_corr_with_y", ascending=False)

CORR_THRESHOLD, MISSING_THRESHOLD = 0.02, 0.5
selected = selection_table[(selection_table["abs_corr_with_y"] >= CORR_THRESHOLD) &
                            (selection_table["missing_rate"] < MISSING_THRESHOLD)].index.tolist()

structural_flags = ["IsHoliday", "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas"]
calendar_cols = ["Month", "WeekOfYear"]
futr_exog_cols = selected + structural_flags + calendar_cols

print("selection-ის კრიტერიუმებს აკმაყოფილებს (corr-based):", selected)
print("+ structural flags:", structural_flags)
print("+ calendar:", calendar_cols)
print("\nსაბოლოო futr_exog_list TFT-სთვის:", futr_exog_cols)
selection_table

selection-ის კრიტერიუმებს აკმაყოფილებს (corr-based): ['MarkDown5', 'MarkDown1', 'MarkDown3', 'MarkDown4', 'Unemployment', 'CPI', 'MarkDown2']
+ structural flags: ['IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas']
+ calendar: ['Month', 'WeekOfYear']

საბოლოო futr_exog_list TFT-სთვის: ['MarkDown5', 'MarkDown1', 'MarkDown3', 'MarkDown4', 'Unemployment', 'CPI', 'MarkDown2', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'Month', 'WeekOfYear']


,abs_corr_with_y,missing_rate
MarkDown5,0.050465,0.0
MarkDown1,0.047172,0.0
MarkDown3,0.038562,0.0
MarkDown4,0.037467,0.0
Unemployment,0.025860,0.0
CPI,0.020923,0.0
MarkDown2,0.020720,0.0
Temperature,0.002312,0.0
Fuel_Price,0.000121,0.0


In [9]:
# Static exog: მაღაზიის დონის უცვლელი მახასიათებლები
stores_static = pd.get_dummies(stores, columns=["Type"], prefix="Type", dtype=float)
size_mean, size_std = stores_static["Size"].mean(), stores_static["Size"].std()
stores_static["Size"] = (stores_static["Size"] - size_mean) / size_std

static_cols = [c for c in stores_static.columns if c != "Store"]
print("static_exog_list TFT-სთვის:", static_cols)
stores_static.head()

static_exog_list TFT-სთვის: ['Size', 'Type_A', 'Type_B', 'Type_C']


,Store,Size,Type_A,Type_B,Type_C
0,1,0.329453,1.0,0.0,0.0
1,2,1.128384,1.0,0.0,0.0
2,3,-1.455467,0.0,1.0,0.0
3,4,1.184098,1.0,0.0,0.0
4,5,-1.494903,0.0,1.0,0.0


In [10]:
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="feature_selection", name=f"{MODEL_NAME}_FeatureSelection")
wandb.config.update({"corr_threshold": CORR_THRESHOLD, "missing_threshold": MISSING_THRESHOLD,
                      "futr_exog_list": ",".join(futr_exog_cols), "stat_exog_list": ",".join(static_cols)})
run.finish()

## 4. Train / Validation Split + WMAE + Exog ცხრილები

Test set 39 კვირით მომავალშია, ამიტომ validation-ადაც ბოლო **39 კვირას** ვჭრით
(Kaggle leaderboard-ის რეალური იმიტაცია — იგივე ლოგიკა, რაც NBEATS notebook-ში).
დამატებით ვაგებთ `static_df`-ს (ერთი row თითო `unique_id`-ზე) და
`build_futr_df()` დამხმარე ფუნქციას, რომელიც ნებისმიერი `(unique_id, ds)`
წყვილებისთვის აწყობს მომავალში ცნობილ exog ცვლადებს `store_exog`-იდან.

In [21]:
bool_futr_cols = [c for c in futr_exog_cols if train_fe[c].dtype == bool]
train_fe[bool_futr_cols] = train_fe[bool_futr_cols].astype(int)
store_exog[bool_futr_cols] = store_exog[bool_futr_cols].astype(int)

nf_cols = ["unique_id", "ds", "y", "Store"] + futr_exog_cols
nf_df = train_fe[nf_cols].dropna(subset=["y"] + futr_exog_cols).sort_values(["unique_id", "ds"]).reset_index(drop=True)

cutoff_date = nf_df["ds"].max() - pd.Timedelta(weeks=HORIZON)
Y_train = nf_df[nf_df["ds"] <= cutoff_date].copy()
Y_valid = nf_df[nf_df["ds"] > cutoff_date].copy()

# მხოლოდ საკმარისი ისტორიის მქონე სერიები (>= 2*HORIZON კვირა) და validation-ში არსებული
counts = Y_train.groupby("unique_id").size()
valid_ids = (set(Y_valid["unique_id"]) & set(Y_train["unique_id"]))
valid_ids = [uid for uid in valid_ids if counts.get(uid, 0) >= 2 * HORIZON]

# row-count საკმარისი არ არის: დუბლირებული (unique_id, ds) რიგმა შეიძლება ნამდვილ
# გამოტოვებულ კვირას "წონასწორობით" დაფაროს (count მაინც = HORIZON გამოვა).
# ამიტომ ვამოწმებთ ზუსტ თარიღების სიმრავლეს, არა უბრალოდ რაოდენობას.
expected_valid_dates = set(pd.date_range(
    start=cutoff_date + pd.Timedelta(weeks=1), end=nf_df["ds"].max(), freq="W-FRI"
))
valid_date_sets = Y_valid.groupby("unique_id")["ds"].apply(set)
n_before_horizon_filter = len(valid_ids)
valid_ids = [uid for uid in valid_ids if valid_date_sets.get(uid, set()) == expected_valid_dates]
print(f"არასრული/დუბლირებული calendar coverage-ის გამო გამოირიცხა: "
      f"{n_before_horizon_filter - len(valid_ids)} სერია (დარჩა: {len(valid_ids)})")

Y_train = Y_train[Y_train["unique_id"].isin(valid_ids)].reset_index(drop=True)
Y_valid = Y_valid[Y_valid["unique_id"].isin(valid_ids)].reset_index(drop=True)
nf_df = nf_df[nf_df["unique_id"].isin(valid_ids)].reset_index(drop=True)
print(f"cutoff={cutoff_date.date()} | series={len(valid_ids)} | train rows={len(Y_train)} | valid rows={len(Y_valid)}")

def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))

არასრული/დუბლირებული calendar coverage-ის გამო გამოირიცხა: 131 სერია (დარჩა: 2727)
cutoff=2012-01-27 | series=2727 | train rows=283141 | valid rows=106353


In [12]:
# static_df -- ერთი row თითო unique_id-ზე (Dept-ები ერთი და იმავე Store-ის ფარგლებში static მახასიათებელს იზიარებენ)
all_uids = pd.DataFrame({"unique_id": sorted(valid_ids)})
all_uids["Store"] = all_uids["unique_id"].str.split("_").str[0].astype(int)
static_df = all_uids.merge(stores_static, on="Store", how="left").drop(columns=["Store"])
print("static_df:", static_df.shape)

def build_futr_df(ids_ds_df: pd.DataFrame) -> pd.DataFrame:
    df = ids_ds_df[["unique_id", "ds"]].drop_duplicates().copy()
    df["Store"] = df["unique_id"].str.split("_").str[0].astype(int)
    out = df.merge(store_exog[["Store", "ds"] + futr_exog_cols], on=["Store", "ds"], how="left")
    return out.drop(columns=["Store"])

# Y_train-შიც უკვე გვაქვს Store სვეტი (nf_cols-იდან) -- ვშლით, nf.fit()-ს არ სჭირდება
Y_train = Y_train.drop(columns=["Store"])
Y_valid_for_fit = Y_valid.drop(columns=["Store"])
nf_df = nf_df.drop(columns=["Store"])

static_df: (2858, 5)


## 5. Hyperparameter Search — Stage 1 (Random Search)

იგივე მიდგომა, რაც NBEATS notebook-ში: `search_space`-დან შემთხვევით
ვსემფლავთ `N_TRIALS` უნიკალურ კონფიგურაციას. TFT-სთვის ბიუჯეტი ოდნავ
პატარაა (8 trial ნაცვლად 10-ისა), რადგან attention + LSTM ყოველ ბიჯზე
მნიშვნელოვნად უფრო ძვირია ვიდრე NBEATS-ის fully-connected ბლოკები.

In [13]:
HP_SEARCH_SAMPLE_N = 150

rng_ids = np.random.RandomState(RANDOM_SEED)
sample_ids = rng_ids.choice(sorted(valid_ids), size=min(HP_SEARCH_SAMPLE_N, len(valid_ids)), replace=False)
Y_train_sample = Y_train[Y_train["unique_id"].isin(sample_ids)].reset_index(drop=True)
Y_valid_sample = Y_valid_for_fit[Y_valid_for_fit["unique_id"].isin(sample_ids)].reset_index(drop=True)
static_df_sample = static_df[static_df["unique_id"].isin(sample_ids)].reset_index(drop=True)
print("HP-search series:", len(sample_ids))

import random

search_space = {
    "input_size": [HORIZON, 2 * HORIZON, 3 * HORIZON],
    "hidden_size": [16, 32],
    "learning_rate": [1e-3, 5e-4, 3e-3],
    "batch_size": [32, 64],
}
rng = random.Random(RANDOM_SEED)
N_TRIALS = 8
configs = []
seen = set()
while len(configs) < N_TRIALS:
    cfg = {k: rng.choice(v) for k, v in search_space.items()}
    key = tuple(sorted(cfg.items()))
    if key not in seen:
        seen.add(key)
        configs.append(cfg)
print(f"{len(configs)} random configs to test -- no wall-clock cap, early_stop_patience_steps "
      f"(+max_steps=100_000 as the formal ceiling) controls training length.")

HP-search series: 150
8 random configs to test -- no wall-clock cap, early_stop_patience_steps (+max_steps=100_000 as the formal ceiling) controls training length.


In [14]:
def build_tft(cfg, h):
    return TFT(
        h=h, input_size=cfg["input_size"],
        hidden_size=cfg["hidden_size"],
        futr_exog_list=futr_exog_cols, stat_exog_list=static_cols, hist_exog_list=[],
        loss=MAE(), learning_rate=cfg["learning_rate"],
        batch_size=cfg["batch_size"], windows_batch_size=cfg["batch_size"] * 8,
        max_steps=100_000,  # პრაქტიკულად არასდროს იჭერს -- early_stop_patience_steps აკონტროლებს
        early_stop_patience_steps=5, val_check_steps=50,
        # long-context კონფიგებისთვის ბევრ სერიას არ ჰყოფნის pre-cutoff ისტორია --
        # start_padding_enabled ნულებით ავსებს დასაწყისს (იგივე მიზეზი, რაც NBEATS-ში).
        start_padding_enabled=True,
        scaler_type="robust", random_seed=RANDOM_SEED,
        enable_progress_bar=False,
        # Kaggle-ის 2 GPU-ს ავტომატური DDP-ის თავიდან ასარიდებლად ერთ GPU-ზე ვჭერთ
        # (იგივე მიზეზი, რაც NBEATS-ის build_nbeats-ში).
        accelerator="auto", devices=1,
    )

def evaluate(nf, valid_df):
    futr = build_futr_df(valid_df[["unique_id", "ds"]])
    preds = nf.predict(futr_df=futr)
    model_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
    merged = preds.merge(valid_df[["unique_id", "ds", "y", "IsHoliday"]], on=["unique_id", "ds"], how="inner")
    score = wmae(merged["y"], merged[model_col], merged["IsHoliday"])
    mae = float(np.mean(np.abs(merged["y"] - merged[model_col])))
    return score, mae

In [15]:
results = []
for i, cfg in enumerate(configs):
    trial_label = f"trial{i}"
    print(f"\n=== {MODEL_NAME}_HPSearch_{trial_label} {cfg} ===")
    t0 = time.time()
    run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="hp_search",
                      name=f"{MODEL_NAME}_HPSearch_{trial_label}", config=cfg, reinit=True)

    model = build_tft(cfg, h=HORIZON)
    nf = NeuralForecast(models=[model], freq="W-FRI")
    nf.fit(df=Y_train_sample, static_df=static_df_sample, val_size=VAL_SIZE)

    elapsed_min = (time.time() - t0) / 60
    try:
        score, mae = evaluate(nf, Y_valid_sample)
    except Exception as e:
        print("eval failed:", repr(e))
        score, mae = np.inf, np.inf

    wandb.log({"val_WMAE": score, "val_MAE": mae, "train_minutes": elapsed_min})
    run.finish()
    results.append({**cfg, "trial": trial_label, "val_WMAE": score, "val_MAE": mae, "train_minutes": elapsed_min})
    print(f"WMAE={score:.2f} | MAE={mae:.2f} | {elapsed_min:.1f} min")

results_df = pd.DataFrame(results).sort_values("val_WMAE")
n_failed = int(np.isinf(results_df["val_WMAE"]).sum())
if n_failed:
    print(f"WARNING: {n_failed}/{len(results_df)} trials failed evaluation (val_WMAE=inf) -- check errors above.")

best_row = results_df.iloc[0]
best_cfg = {k: (best_row[k].item() if hasattr(best_row[k], "item") else best_row[k]) for k in search_space}
print("\nStage 1 საუკეთესო კონფიგურაცია:", best_row["trial"], dict(best_cfg))
results_df


=== TFT_HPSearch_trial0 {'input_size': 117, 'hidden_size': 16, 'learning_rate': 0.001, 'batch_size': 64} ===


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 608    | train
7  | static_encoder          | StaticCovariateEncoder   | 10.9 K | train
8  | temporal_encoder        | TemporalCovariateEncoder | 53.7 K | train
9  | temporal_fusion_decoder | TemporalFusionDecode

eval failed: ValueError('There are missing combinations of ids and times in `futr_df`.\nYou can run the `make_future_dataframe()` method to get the expected combinations or the `get_missing_future(futr_df)` method to get the missing combinations.')


train_minutes,▁
+2,...
train_minutes,1.27919
val_MAE,inf
val_WMAE,inf


...

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 608    | train
7  | static_encoder          | StaticCovariateEncoder   | 10.9 K | train
8  | temporal_encoder        | TemporalCovariateEncoder | 53.7 K | train
9  | temporal_fusion_decoder | TemporalFusionDecode

eval failed: ValueError('There are missing combinations of ids and times in `futr_df`.\nYou can run the `make_future_dataframe()` method to get the expected combinations or the `get_missing_future(futr_df)` method to get the missing combinations.')


train_minutes,▁
+2,...
train_minutes,0.57157
val_MAE,inf
val_WMAE,inf


WMAE=inf | MAE=inf | 0.6 min

Stage 1 საუკეთესო კონფიგურაცია: trial0 {'input_size': 117, 'hidden_size': 16, 'learning_rate': 0.001, 'batch_size': 64}


,input_size,hidden_size,learning_rate,batch_size,trial,val_WMAE,val_MAE,train_minutes
0,117,16,0.0010,64,trial0,inf,inf,1.279195
1,39,16,0.0010,32,trial1,inf,inf,0.587294
2,117,16,0.0030,64,trial2,inf,inf,1.027382
3,39,16,0.0030,32,trial3,inf,inf,0.610408
4,117,32,0.0010,64,trial4,inf,inf,1.506223
5,117,32,0.0010,32,trial5,inf,inf,0.853953
6,117,32,0.0005,64,trial6,inf,inf,2.102178
7,39,16,0.0005,32,trial7,inf,inf,0.571565


## 5b. Hyperparameter Search — Stage 2 (Full Factorial Grid Search)

`input_size`-ის კანდიდატები **დინამიურად** გამოითვლება Stage 1-ის საუკეთესო
trial-ებიდან (ტოპ-3-ის უნიკალური მნიშვნელობები), + დამატებით `52` (ერთი სრული
სეზონური წელი), თუ ჯერ არ არის სიაში — იგივე domain-motivated არჩევანი, რაც
NBEATS notebook-ში. **`hidden_size`/`learning_rate`/`batch_size`-ის კანდიდატები
ჯერ არ არის დაზუსტებული Stage 1-ის კონკრეტულ შედეგებზე** (ეს notebook ჯერ არ
გაშვებულა) — Stage 1-ის დასრულების შემდეგ გადახედეთ `results_df`-ს და, თუ
საჭიროა, ხელით დაარედაქტირეთ `search_space_2` ქვემოთ (მაგ. თუ `learning_rate`
ან `batch_size` რომელიმე მნიშვნელობა ცალსახად ცუდია, ჩაანაცვლეთ ახლით,
NBEATS-ის Stage 2-ის მსგავსად).

In [16]:
import itertools

# input_size: დინამიური, Stage 1-ის ტოპ-3-დან + 52 (სრული სეზონური წელი) დომენური დამატებით
top_input_sizes = sorted(set(results_df.sort_values("val_WMAE").head(3)["input_size"].tolist()))
if 52 not in top_input_sizes:
    top_input_sizes = sorted(top_input_sizes + [52])
print("Stage 2 input_size candidates (dynamic):", top_input_sizes)

search_space_2 = {
    "input_size": top_input_sizes,
    "hidden_size": [16, 32],
    "learning_rate": [1e-3, 2e-3, 4e-3],
    "batch_size": [32, 64],
}
# შენიშვნა: Stage 1-ის შედეგების მიხედვით ხელით დაარედაქტირეთ ზემოთ, თუ საჭიროა
# (მაგ. თუ learning_rate=4e-3 ან 1e-3 ცალსახად ცუდია, NBEATS-ის Stage 2-ის ანალოგიურად).

keys, values = zip(*search_space_2.items())
configs_2 = [dict(zip(keys, combo)) for combo in itertools.product(*values)]
print(f"{len(configs_2)} grid configs to test (Stage 2 -- full factorial).")

Stage 2 input_size candidates (dynamic): [39, 52, 117]
36 grid configs to test (Stage 2 -- full factorial).


In [17]:
results_2 = []
for i, cfg in enumerate(configs_2):
    trial_label = f"trial{i}"
    print(f"\n=== {MODEL_NAME}_HPSearch2_{trial_label} {cfg} ===")
    t0 = time.time()
    run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="hp_search_2",
                      name=f"{MODEL_NAME}_HPSearch2_{trial_label}", config=cfg, reinit=True)

    model = build_tft(cfg, h=HORIZON)
    nf = NeuralForecast(models=[model], freq="W-FRI")
    nf.fit(df=Y_train_sample, static_df=static_df_sample, val_size=VAL_SIZE)

    elapsed_min = (time.time() - t0) / 60
    try:
        score, mae = evaluate(nf, Y_valid_sample)
    except Exception as e:
        print("eval failed:", repr(e))
        score, mae = np.inf, np.inf

    wandb.log({"val_WMAE": score, "val_MAE": mae, "train_minutes": elapsed_min})
    run.finish()
    results_2.append({**cfg, "trial": trial_label, "val_WMAE": score, "val_MAE": mae, "train_minutes": elapsed_min})
    print(f"WMAE={score:.2f} | MAE={mae:.2f} | {elapsed_min:.1f} min")

results_2_df = pd.DataFrame(results_2).sort_values("val_WMAE")
n_failed_2 = int(np.isinf(results_2_df["val_WMAE"]).sum())
if n_failed_2:
    print(f"WARNING: {n_failed_2}/{len(results_2_df)} trials failed evaluation (val_WMAE=inf) -- check errors above.")

best_row_2 = results_2_df.iloc[0]
best_cfg_2 = {k: (best_row_2[k].item() if hasattr(best_row_2[k], "item") else best_row_2[k]) for k in search_space_2}
print("\nStage 2 საუკეთესო კონფიგურაცია:", best_row_2["trial"], dict(best_cfg_2))
results_2_df


=== TFT_HPSearch2_trial0 {'input_size': 39, 'hidden_size': 16, 'learning_rate': 0.001, 'batch_size': 32} ===


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 608    | train
7  | static_encoder          | StaticCovariateEncoder   | 10.9 K | train
8  | temporal_encoder        | TemporalCovariateEncoder | 53.7 K | train
9  | temporal_fusion_decoder | TemporalFusionDecode

eval failed: ValueError('There are missing combinations of ids and times in `futr_df`.\nYou can run the `make_future_dataframe()` method to get the expected combinations or the `get_missing_future(futr_df)` method to get the missing combinations.')


train_minutes,▁
+2,...
train_minutes,0.57067
val_MAE,inf
val_WMAE,inf


WMAE=inf | MAE=inf | 0.6 min

=== TFT_HPSearch2_trial1 {'input_size': 39, 'hidden_size': 16, 'learning_rate': 0.001, 'batch_size': 64} ===


...

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 1.2 K  | train
7  | static_encoder          | StaticCovariateEncoder   | 41.3 K | train
8  | temporal_encoder        | TemporalCovariateEncoder | 192 K  | train
9  | temporal_fusion_decoder | TemporalFusionDecode

eval failed: ValueError('There are missing combinations of ids and times in `futr_df`.\nYou can run the `make_future_dataframe()` method to get the expected combinations or the `get_missing_future(futr_df)` method to get the missing combinations.')


train_minutes,▁
+2,...
train_minutes,1.51258
val_MAE,inf
val_WMAE,inf


WMAE=inf | MAE=inf | 1.5 min

Stage 2 საუკეთესო კონფიგურაცია: trial0 {'input_size': 39, 'hidden_size': 16, 'learning_rate': 0.001, 'batch_size': 32}


,input_size,hidden_size,learning_rate,batch_size,trial,val_WMAE,val_MAE,train_minutes
0,39,16,0.001,32,trial0,inf,inf,0.570674
1,39,16,0.001,64,trial1,inf,inf,0.581852
2,39,16,0.002,32,trial2,inf,inf,0.418895
3,39,16,0.002,64,trial3,inf,inf,0.518186
4,39,16,0.004,32,trial4,inf,inf,0.609595
5,39,16,0.004,64,trial5,inf,inf,0.713077
6,39,32,0.001,32,trial6,inf,inf,0.637461
7,39,32,0.001,64,trial7,inf,inf,1.216485
8,39,32,0.002,32,trial8,inf,inf,0.790403
9,39,32,0.002,64,trial9,inf,inf,0.720155


In [22]:
# Stage 1 vs Stage 2 -- საბოლოო გამარჯვებული კონფიგურაციის შერჩევა
if best_row_2["val_WMAE"] < best_row["val_WMAE"]:
    winning_stage = "stage_2 (full grid)"
    best_cfg = best_cfg_2
    best_val_wmae = float(best_row_2["val_WMAE"])
else:
    winning_stage = "stage_1 (random search)"
    best_val_wmae = float(best_row["val_WMAE"])
    # best_cfg რჩება stage 1-ის მნიშვნელობაზე

print(f"გამარჯვებული: {winning_stage}")
print(f"საბოლოო best_cfg: {best_cfg}")
print(f"საბოლოო HP-search val_WMAE (150-series sample): {best_val_wmae:.2f}")
print(f"შედარებისთვის -- stage1 best: {best_row['val_WMAE']:.2f} | stage2 best: {best_row_2['val_WMAE']:.2f}")

გამარჯვებული: stage_1 (random search)
საბოლოო best_cfg: {'input_size': 117, 'hidden_size': 16, 'learning_rate': 0.001, 'batch_size': 64}
საბოლოო HP-search val_WMAE (150-series sample): inf
შედარებისთვის -- stage1 best: inf | stage2 best: inf


## 6. Final Training

საბოლოო, ორივე HP-search ეტაპიდან გამარჯვებული კონფიგურაციით ვარჯიშდება
მთელ `Y_train`-ზე (არა მხოლოდ 150-სერიიან სემფლზე) და ფასდება `Y_valid`-ზე
(39-კვირიანი holdout, არასოდეს ნანახი) -- ეს არის საბოლოო რიცხვი დანარჩენ
არქიტექტურებთან (NBEATS-ის ჩათვლით) შედარებისთვის.

In [23]:
static_df_train = static_df[static_df["unique_id"].isin(set(Y_train["unique_id"]))].reset_index(drop=True)

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="final",
                  name=f"{MODEL_NAME}_Final_Training",
                  config=dict(best_cfg))
wandb.config.update({"winning_hp_search_stage": winning_stage})

t0 = time.time()
final_model = build_tft(best_cfg, h=HORIZON)
nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=Y_train, static_df=static_df_train, val_size=VAL_SIZE)
elapsed_min = (time.time() - t0) / 60

final_wmae, final_mae = evaluate(nf_final, Y_valid_for_fit)
wandb.log({"val_WMAE": final_wmae, "val_MAE": final_mae, "train_minutes": elapsed_min})
wandb.config.update({"n_series_used": len(valid_ids)})
run.finish()

print(f"TFT final WMAE={final_wmae:.2f} | MAE={final_mae:.2f} | {elapsed_min:.1f} min")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 608    | train
7  | static_encoder          | StaticCovariateEncoder   | 10.9 K | train
8  | temporal_encoder        | TemporalCovariateEncoder | 53.7 K | train
9  | temporal_fusion_decoder | TemporalFusionDecode

ValueError: There are missing combinations of ids and times in `futr_df`.
You can run the `make_future_dataframe()` method to get the expected combinations or the `get_missing_future(futr_df)` method to get the missing combinations.

## 7. Pipeline + Model Registry

`nf_final` ვარჯიშობდა მხოლოდ `Y_train`-ზე (cutoff-მდე) — განზრახ, რომ
`Y_valid`-ზე (39 კვირა, არასოდეს უნახავს) პატიოსანი `final_wmae` გამოგვეთვალა.
Deployment-ისთვის იმავე `best_cfg`-ით კიდევ ერთხელ ვარჯიშდება მოდელი **მთელ**
ხელმისაწვდომ ისტორიაზე (`nf_df` = `Y_train` + `Y_valid`) და იხვევა
`TFTPipeline`-ში (raw dataframe → `clean_data` → `engineer_features` →
`build_futr_df` → TFT პროგნოზი — ეშვება პირდაპირ raw `test.csv`-ზე), რომელიც
შემდეგ Wandb Registry-ში რეგისტრირდება.

**შენიშვნა coverage-ის შესახებ** (იგივე, რაც NBEATS-ში): `valid_ids`-ის
ფილტრი გამორიცხავს არასაკმარისი ისტორიის მქონე Store-Dept წყვილებს. ამ
სერიებისთვის `TFTPipeline.predict()`-ს აქვს **fallback**: Store-Dept
ისტორიული საშუალო, ან Store საშუალო, ან გლობალური საშუალო.

In [ ]:
# Deployment refit მთელ ხელმისაწვდომ ისტორიაზე (არა მხოლოდ Y_train)
static_df_deploy = static_df[static_df["unique_id"].isin(set(nf_df["unique_id"]))].reset_index(drop=True)

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="deploy_refit",
                  name=f"{MODEL_NAME}_Deploy_Refit",
                  config=dict(best_cfg))

t0 = time.time()
deploy_model = build_tft(best_cfg, h=HORIZON)
nf_deploy = NeuralForecast(models=[deploy_model], freq="W-FRI")
nf_deploy.fit(df=nf_df, static_df=static_df_deploy, val_size=VAL_SIZE)
elapsed_min = (time.time() - t0) / 60

wandb.log({"train_minutes": elapsed_min})
wandb.config.update({"deploy_max_train_date": str(nf_df["ds"].max().date())})

PIPELINE_DIR = "./tft_pipeline_model"  # ერთადერთი შენახული მოდელის დირექტორია ამ notebook-ში
nf_deploy.save(path=PIPELINE_DIR, overwrite=True)
run.finish()
print(f"Deploy refit done ({elapsed_min:.1f} min), max train date={nf_df['ds'].max().date()}")

In [ ]:
class TFTPipeline:
    '''raw dataframe -> clean_data -> engineer_features -> build_futr_df -> TFT პროგნოზი.
    ეშვება პირდაპირ raw (დაუმუშავებელ) test set-ზე. TFT-ს, NBEATS-ისგან განსხვავებით,
    predict()-ისთვის სჭირდება futr_df (მომავალში ცნობილი exog ცვლადები) -- ეს აიგება
    store_exog-იდან, რომელიც features.csv-დან მომდინარეობს და test-პერიოდსაც ფარავს.

    Fallback: TFT-სთვის უცნობი (valid_ids-ის ფილტრით გამორიცხული) სერიები ივსება
    Store-Dept ისტორიული საშუალოთი (train_fe-დან), ან Store საშუალოთი, ან
    გლობალური საშუალოთი -- რომ ყველა test.csv-ის row-ს ჰქონდეს პროგნოზი.'''

    def __init__(self, nf, history_df):
        self.nf = nf
        self._dept_mean = history_df.groupby(["Store", "Dept"])["y"].mean()
        self._store_mean = history_df.groupby("Store")["y"].mean()
        self._global_mean = float(history_df["y"].mean())

    def _fallback_value(self, store, dept):
        key = (store, dept)
        if key in self._dept_mean.index:
            return float(self._dept_mean[key])
        if store in self._store_mean.index:
            return float(self._store_mean[store])
        return self._global_mean

    def predict(self, raw_df: pd.DataFrame) -> pd.DataFrame:
        df = raw_df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df, _ = clean_data(merge_sources(df), is_train=False)
        df = engineer_features(df)

        futr = build_futr_df(df[["unique_id", "ds"]])
        preds = self.nf.predict(futr_df=futr)
        model_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
        out = df[["unique_id", "ds", "Store", "Dept"]].merge(preds, on=["unique_id", "ds"], how="left")
        out = out.rename(columns={model_col: "Weekly_Sales_Pred"})

        missing = out["Weekly_Sales_Pred"].isna()
        n_missing = int(missing.sum())
        if n_missing:
            out.loc[missing, "Weekly_Sales_Pred"] = out.loc[missing].apply(
                lambda r: self._fallback_value(r["Store"], r["Dept"]), axis=1
            )
            print(f"TFTPipeline: filled {n_missing}/{len(out)} rows "
                  f"({n_missing / len(out):.1%}) via Store/Dept fallback mean "
                  f"(series not seen by TFT due to valid_ids history filter).")

        return out.drop(columns=["Store", "Dept"])


pipeline = TFTPipeline(nf_deploy, history_df=train_fe)
preds_preview = pipeline.predict(test_raw)  # რეალური, დაუმუშავებელი test.csv
print(preds_preview.shape)
preds_preview.head()

In [ ]:
# რეგისტრაცია Wandb Registry-ში (wandb-ის ეკვივალენტი MLflow-ის Model Registry-ის).
# ახალი Registry schema: target_path = `wandb-registry-{REGISTRY_NAME}/{COLLECTION_NAME}`
# (ძველი `{entity}/model-registry/{name}` schema მიგრირებულ organizations-ზე HTTP 400-ს
# აგდებს). "model" ორგანიზაციის ახალ Registry სისტემაში ნაგულისხმევად უნდა არსებობდეს --
# თუ არა, საჭიროა წინასწარ შექმნა UI-დან (Registry ტაბი) ან wandb.Api().create_registry(...)-ით.
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="registry",
                  name=f"{MODEL_NAME}_Pipeline_Registry")

artifact = wandb.Artifact(name=f"{MODEL_NAME}_pipeline", type="model", metadata=dict(best_cfg))
artifact.add_dir(PIPELINE_DIR)
run.log_artifact(artifact)
run.link_artifact(artifact, target_path=f"wandb-registry-model/{MODEL_NAME}")
run.finish()
print(f"'{MODEL_NAME}_pipeline' დარეგისტრირდა Wandb Registry-ში.")

## 8. შედეგების ვიზუალიზაცია (README-სთვის)

5 გრაფიკი, PNG-ად შენახული `./tft_figures/`-ში — feature selection-ის corr
(ახლა რეალურად გამოყენებული ცვლადებისთვის), Stage 1 (random search) და
Stage 2 (grid search) WMAE შედარებები, actual-vs-predicted სემფლ სერიებზე
(holdout) და predicted-vs-actual scatter (calibration). ყველა აგებულია
`nf_final`-ის (Y_train-ზე გავარჯიშებული, Y_valid-ზე არასოდეს ნანახი)
პროგნოზებზე.

In [ ]:
import matplotlib.pyplot as plt

FIGURES_DIR = "./tft_figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# 1) Feature Selection -- |corr(feature, y)| (ამჯერად ეს ცვლადები რეალურად ეძლევა TFT-ს)
sel_sorted = selection_table.sort_values("abs_corr_with_y")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(sel_sorted.index, sel_sorted["abs_corr_with_y"], color="#C44E52")
ax.axvline(CORR_THRESHOLD, color="red", linestyle="--", label=f"threshold={CORR_THRESHOLD}")
ax.set_xlabel("|corr(feature, y)|")
ax.set_title("TFT — Exogenous Correlation (these ARE fed to the model as futr_exog_list)")
ax.legend()
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/tft_feature_selection.png", dpi=150)
plt.show()

In [ ]:
# 2) Stage 1 -- Random Search WMAE per trial (best highlighted)
plot_df = results_df.sort_values("val_WMAE")
colors = ["#55A868" if t == best_row["trial"] else "#C44E52" for t in plot_df["trial"]]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_df["trial"], plot_df["val_WMAE"], color=colors)
ax.set_xlabel("Validation WMAE")
ax.set_title("TFT — Stage 1: Random Search Results (best trial in green)")
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/tft_hp_search1_results.png", dpi=150)
plt.show()

In [ ]:
# 3) Stage 2 -- Full Grid Search WMAE per trial (best highlighted)
plot_df2 = results_2_df.sort_values("val_WMAE")
colors2 = ["#55A868" if t == best_row_2["trial"] else "#C44E52" for t in plot_df2["trial"]]
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(plot_df2["trial"], plot_df2["val_WMAE"], color=colors2)
ax.set_xlabel("Validation WMAE")
ax.set_title(f"TFT — Stage 2: Full Grid Search Results (best trial in green, winner: {winning_stage})")
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/tft_hp_search2_results.png", dpi=150)
plt.show()

In [ ]:
# nf_final-ის პროგნოზები Y_valid-ზე (holdout, არასოდეს ნანახი) -- შემდეგი ორი გრაფიკისთვის
futr_valid = build_futr_df(Y_valid_for_fit[["unique_id", "ds"]])
preds_valid = nf_final.predict(futr_df=futr_valid)
model_col = [c for c in preds_valid.columns if c not in ("unique_id", "ds")][0]
merged_valid = preds_valid.merge(Y_valid_for_fit[["unique_id", "ds", "y"]], on=["unique_id", "ds"], how="inner")

# 4) Actual vs Predicted -- 4 შემთხვევით შერჩეული სერია validation window-ზე
sample_uids = merged_valid["unique_id"].drop_duplicates().sample(4, random_state=RANDOM_SEED).tolist()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, uid in zip(axes.flat, sample_uids):
    sub = merged_valid[merged_valid["unique_id"] == uid].sort_values("ds")
    ax.plot(sub["ds"], sub["y"], label="Actual", marker="o")
    ax.plot(sub["ds"], sub[model_col], label="Predicted", marker="x")
    ax.set_title(uid)
    ax.tick_params(axis="x", rotation=45)
axes.flat[0].legend()
fig.suptitle("TFT — Actual vs Predicted (validation holdout, sample series)")
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/tft_actual_vs_predicted.png", dpi=150)
plt.show()

In [ ]:
# 5) Predicted vs Actual scatter (calibration) -- მთელ validation holdout-ზე
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(merged_valid["y"], merged_valid[model_col], alpha=0.15, s=8, color="#C44E52")
lims = [0, max(merged_valid["y"].max(), merged_valid[model_col].max())]
ax.plot(lims, lims, color="red", linestyle="--", label="y = ŷ")
ax.set_xlabel("Actual Weekly_Sales")
ax.set_ylabel("Predicted Weekly_Sales")
ax.set_title("TFT — Predicted vs Actual (validation holdout)")
ax.legend()
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/tft_pred_vs_actual_scatter.png", dpi=150)
plt.show()

print("Figures saved to:", os.path.abspath(FIGURES_DIR))

## 9. Full Results Summary (ტექსტური შეჯამება README-სთვის)

ყველა ეტაპის შედეგი ერთ ადგილას, ტექსტად — პირდაპირ ჩასასმელად `README.md`-ში.
ასევე ინახება `./tft_results_summary.txt` ფაილში.

In [ ]:
import contextlib, io

def print_summary():
    print("=" * 70)
    print(f"{MODEL_NAME} — FULL RESULTS SUMMARY")
    print("=" * 70)

    print("\n--- 1. Data Cleaning ---")
    for k, v in train_stats.items():
        print(f"  train_{k}: {v}")
    for k, v in test_stats.items():
        print(f"  test_{k}: {v}")

    print("\n--- 2. Feature Engineering ---")
    print(f"  n_rows_train: {len(train_fe)}")
    print(f"  n_unique_series: {train_fe['unique_id'].nunique()}")
    print(f"  store_exog date range: {store_exog['ds'].min().date()} -> {store_exog['ds'].max().date()}")

    print("\n--- 3. Feature Selection (used as TFT input) ---")
    print(selection_table.to_string())
    print(f"  futr_exog_list: {futr_exog_cols}")
    print(f"  stat_exog_list: {static_cols}")

    print("\n--- 4. Train/Valid Split ---")
    print(f"  cutoff_date: {cutoff_date.date()}")
    print(f"  n_series (valid_ids): {len(valid_ids)}")
    print(f"  train rows: {len(Y_train)} | valid rows: {len(Y_valid_for_fit)}")

    print("\n--- 5. Hyperparameter Search -- Stage 1 (Random Search) ---")
    print(results_df.to_string(index=False))
    print(f"\n  Stage 1 best: {best_row['trial']} | val_WMAE={best_row['val_WMAE']:.2f} | val_MAE={best_row['val_MAE']:.2f}")

    print(f"\n--- 5b. Hyperparameter Search -- Stage 2 (Full Grid Search, {len(configs_2)} trials) ---")
    print(results_2_df.to_string(index=False))
    print(f"\n  Stage 2 best: {best_row_2['trial']} | val_WMAE={best_row_2['val_WMAE']:.2f} | val_MAE={best_row_2['val_MAE']:.2f}")

    print(f"\n  WINNING STAGE: {winning_stage}")
    print(f"  FINAL best_cfg: {best_cfg}")

    print("\n--- 6. Final Training (best_cfg, full Y_train) ---")
    print(f"  final_wmae (holdout, 39-week): {final_wmae:.2f}")
    print(f"  final_mae  (holdout, 39-week): {final_mae:.2f}")

    print("\n--- 7. Deploy Refit (best_cfg, full history nf_df) ---")
    print(f"  max train date used: {nf_df['ds'].max().date()}")
    print(f"  pipeline saved to: {PIPELINE_DIR}")

    print("\n--- 8. Pipeline Preview (raw test.csv -> predictions, with fallback) ---")
    print(f"  shape: {preds_preview.shape}")
    print(preds_preview.head(10).to_string(index=False))

    print("\n--- 9. Wandb Registry ---")
    print(f"  artifact name: {MODEL_NAME}_pipeline")
    print(f"  registry target: wandb-registry-model/{MODEL_NAME}")

    print("\n" + "=" * 70)
    print("END OF SUMMARY")
    print("=" * 70)

print_summary()

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print_summary()
summary_text = buf.getvalue()
with open("./tft_results_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)
print(f"\nSaved to ./tft_results_summary.txt ({len(summary_text)} chars)")

## შედეგი

`final_wmae` არის ის რიცხვი, რომელიც ამ notebook-იდან შედარდება NBEATS-ის და
დანარჩენი არქიტექტურების `val_WMAE`-ს — ყველაზე დაბალი WMAE იგებს. TFT-ის
შემთხვევაში ეს შედარება განსაკუთრებით საინტერესოა, რადგან NBEATS-ისგან
განსხვავებით TFT-მ **რეალურად გამოიყენა** გარე ცვლადები (MarkDown, CPI,
Unemployment, holiday/calendar ცვლადები, მაღაზიის Type/Size) — თუ TFT-ის
WMAE მნიშვნელოვნად სჯობს NBEATS-ს, ეს მტკიცებულებაა, რომ ამ გარე ცვლადებს
რეალური პროგნოზირებადი ღირებულება აქვთ; თუ არა სჯობს (ან უარესია), ეს
საინტერესო თავისთავად შედეგია საბოლოო README-ის/პრეზენტაციის შედარებისთვის
("რატომ" კითხვაზე პასუხი -- overfitting მეტ პარამეტრზე, თუ ნაკლები HP-search
ბიუჯეტი, თუ გარე ცვლადებს რეალურად სუსტი სიგნალი აქვთ, როგორც Feature
Selection-ის corr ცხრილშიც ჩანდა).

საბოლოო `best_cfg` არჩეულია **ორსაფეხურიანი HP search-ით** (Stage 1: random
search 36-დან 8 კონფიგურაცია; Stage 2: სრული ფაქტორული grid search
დინამიურად დაზუსტებულ `input_size`-ის დიაპაზონში). `TFTPipeline`
(deployment-ზე refit-ული მოდელით, `nf_deploy`, **fallback-ით** უცნობი
სერიებისთვის) დარეგისტრირებულია Wandb Registry-ში და მზადაა raw test
set-ზე submission-ის დასაგენერირებლად. `./tft_figures/`-ში შენახული 5 PNG
და `./tft_results_summary.txt` README.md-ში ჩასასმელად მზადაა.